# CONVOLUTIONAL NEURAL NETWORK (CNN)

In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import torchvision
from torchvision.datasets import CIFAR10
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms # Here we can define the number of transformation that we required for the each image for example scaling and normalizatoin ets.


In [4]:
# DATASETS AND DATALOADERS
# image ==> scale (0,1) ==> (-1,1) 
transform = transforms.Compose([
    transforms.ToTensor(), # this will not only transform our images into tensor but also scale the images
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # Here we define the standard deviation and mean values to perform the normalization. these are the set values for this dataset
    
])

In [5]:
trainset = CIFAR10(root="./data", train = True, download=True, transform=transform) # ./data is the location to store the dataset
testset = CIFAR10(root="./data", train = False, download=True, transform=transform)

100%|████████████████████████████████████████| 170M/170M [00:22<00:00, 7.43MB/s]


In [6]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [7]:
testset

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [8]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)


# BUILD THE CNN

In [14]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel and stride value are 2 and 2 

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel and stride value are 2 and 2 

            nn.Conv2d(64,128, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) # kernel and stride value are 2 and 2 
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256), # 256 is a random value of ouput neuron
            nn.ReLU(),

            nn.Linear(256, 10) # because we have the 10 output classses 
        )
    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flatening # .view() reshape tensor into shape (a, b), # x.size(0) So you keep batch size unchanged. # -1 figure out this dimension automatically
        x = self.fc_layers(x)
        return x
        

In [15]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Training the CNN Model

In [19]:
epocs = 10
for epoc in range(epocs):
    epoc_training_loss = 0.0

    for images , labels in trainloader:
        optimizer.zero_grad()
        output = model.forward(images) # FP
        loss = criterion(output, labels)
        loss.backward() # BP 
        optimizer.step() # update Parameters

        epoc_training_loss += loss.item()
    print(f" epoch = {epoc+1} / {epocs} and loss = {epoc_training_loss/len(trainloader)*100}")
    

 epoch = 1 / 10 and loss = 77.05552680108248
 epoch = 2 / 10 and loss = 63.767376134310226
 epoch = 3 / 10 and loss = 53.417499603517825
 epoch = 4 / 10 and loss = 43.82679134683536
 epoch = 5 / 10 and loss = 34.71407165364994
 epoch = 6 / 10 and loss = 27.029880058125155
 epoch = 7 / 10 and loss = 21.260788610390843
 epoch = 8 / 10 and loss = 16.079699133982515
 epoch = 9 / 10 and loss = 13.40076306244106
 epoch = 10 / 10 and loss = 11.97223190314677


In [23]:
# Eval the model 
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in testloader:
        outputs = model.(images)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f" accuracy = {correct/total * 100} %")

 accuracy = 74.45 %
